# Preference-shift measurement pilot — AgentRecBench

Reproduces the Amazon short-term TVD pilot, eligibility checks, Figure 2, and Figure 4. **No recommendation-model accuracy or ablation results are claimed.** The original AgentRecBench dataset is downloaded from Hugging Face. The upstream dataset revision should be pinned before final submission.

In [ ]:
%pip -q install pandas numpy matplotlib huggingface_hub

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, HfApi

REPO = "SGJQovo/AgentRecBench"
REVISION = None  # Replace with the exact upstream commit SHA before submission
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
print("Upstream current revision:", HfApi().dataset_info(REPO).sha)

In [ ]:
def download(filename):
    return hf_hub_download(repo_id=REPO, filename=filename, repo_type="dataset", revision=REVISION)

case = pd.read_csv(download("task/short_term/amazon/amazon_case.csv"))
case["parsed_date"] = pd.to_datetime(case["parsed_date"])
print("Historical interactions:", len(case), "Users:", case.user_id.nunique(), "Items:", case.item_id.nunique())

In [ ]:
# Stream the large JSONL metadata file instead of calling json.load() on the whole file.
item_ids = set(case.item_id)
item_to_category = {}
matched_ids = set()
with open(download("output_data_short/item.json"), encoding="utf-8") as handle:
    for line in handle:
        record = json.loads(line)
        item_id = record.get("item_id")
        if item_id in item_ids:
            matched_ids.add(item_id)
            categories = record.get("categories")
            if isinstance(categories, list) and len(categories) > 1:
                item_to_category[item_id] = categories[1]
print("Matched item IDs:", len(matched_ids), "/", len(item_ids))

In [ ]:
history = case.copy()
history["category"] = history.item_id.map(item_to_category)
history = history.sort_values(["user_id", "parsed_date", "item_id"])
print("Missing second-level categories:", history.category.isna().sum())

def tvd(early, recent):
    p = early.value_counts(normalize=True)
    q = recent.value_counts(normalize=True)
    return 0.5 * sum(abs(p.get(c, 0) - q.get(c, 0)) for c in p.index.union(q.index))

rows = []
for user_id, group in history.groupby("user_id"):
    n = len(group)
    if n < 4:
        continue
    midpoint = n // 2
    early, recent = group.iloc[:midpoint].category, group.iloc[midpoint:].category
    if early.isna().any() or recent.isna().any():
        continue
    rows.append(dict(user_id=user_id, n_interactions=n, past_n=len(early), recent_n=len(recent), shift_score=tvd(early,recent)))
shift_df = pd.DataFrame(rows)
shift_df["history_group"] = np.where(shift_df.n_interactions <= 5, "4–5 interactions", "6+ interactions")
print("Eligible:", len(shift_df), "/", history.user_id.nunique())
print(shift_df.shift_score.describe())
print("TVD = 0:", np.isclose(shift_df.shift_score, 0).sum(), "TVD = 1:", np.isclose(shift_df.shift_score, 1).sum())

In [ ]:
# Export de-identified user-level measurements (not original user IDs).
public = shift_df.drop(columns=["user_id"]).reset_index(drop=True).copy()
public.insert(0, "pilot_user_index", np.arange(1, len(public) + 1))
public.to_csv(OUT / "preference_shift_scores_deidentified.csv", index=False)

summary = shift_df.groupby("history_group").agg(users=("shift_score","size"), mean_tvd=("shift_score","mean"), median_tvd=("shift_score","median"))
print(summary)

In [ ]:
# Figure 2: observed TVD histogram
scores = shift_df.shift_score
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(scores, bins=np.linspace(0, 1, 11), edgecolor="black", alpha=.8)
ax.axvline(scores.mean(), linestyle="--", color="black", label=f"Mean = {scores.mean():.3f}")
ax.set(xlabel="Preference Shift Score (TVD)", ylabel="Number of Users", title=f"Distribution of Category-Based Preference Shifts (n = {len(scores)})")
ax.legend(); fig.tight_layout(); fig.savefig(OUT / "figure2_preference_shift.png", dpi=300, bbox_inches="tight"); plt.show()

In [ ]:
# Figure 4: feasibility and score categories
N = history.user_id.nunique()
eligible = len(shift_df)
zero = int(np.isclose(scores,0).sum())
one = int(np.isclose(scores,1).sum())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, labels, values, denominator, title in [
    (axes[0], ["Eligible", "Ineligible"], [eligible, N-eligible], N, "A. Measurement Eligibility"),
    (axes[1], ["TVD = 0", "0 < TVD < 1", "TVD = 1"], [zero, eligible-zero-one, one], eligible, "B. TVD Score Categories")]:
    bars = ax.bar(labels, values, edgecolor="black", alpha=.75)
    ax.set_title(title); ax.set_ylabel("Number of Users"); ax.set_ylim(0, max(values)*1.25)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, value+0.5, f"{value} ({value/denominator:.1%})", ha="center", fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
fig.tight_layout(); fig.savefig(OUT / "figure4_pilot_feasibility.png", dpi=300, bbox_inches="tight"); plt.show()

## Limitations

TVD is a proxy for **observed category-distribution change**, not latent psychological preference change. Short interaction histories may inflate extreme scores. The pilot does **not** yet test HR@5, model adaptation, or recent-history ablation. Verify the upstream license and redistribution permissions before public dataset release.